# DPO Hyperparameter Sweep for the Pareto-DPO v2 Collapse (Kaggle 2xT4)

The Pareto-DPO v2 training run (`experiments/018_pareto_dpo_v2_extra_rollouts/data/pareto_dpo_pairs_v2.jsonl`, 298 pairs from the full 0-9 rollout pool) hit the trainer's own collapse guard at step 215/298 using `lr=1e-5, beta=0.1` -- the exact hyperparameters every prior DPO run in this project (Full DPO, Pareto-DPO v1) trained cleanly with at the smaller ~134-154-pair scale. This sweeps lower learning rates and higher beta (a tighter trust region around the reference model) around that known-collapsing baseline, which is kept in the grid for a documented before/after comparison. One fresh base-model load per config (no adapter reuse across configs).


In [ ]:
# Cell 1: Check GPU hardware and install sm_60 compatible PyTorch stack if Tesla P100 is assigned
import os, subprocess, sys, torch

print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.12 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print('Packages successfully configured.')


In [ ]:
# Cell 2: Checkout repository and run the sweep
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
# main has scripts/train/sweep_dpo.py -- plain clone defaults to main.
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/sweep_dpo.py',
    '--dataset-path', 'experiments/018_pareto_dpo_v2_extra_rollouts/data/pareto_dpo_pairs_v2.jsonl',
    '--output-dir', '/kaggle/working/qwen_vl_dpo_v2_tuned_adapter',
    '--sweep-output-dir', '/kaggle/working/dpo_sweep',
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 3: Validate output artifacts and print the winning config
import json
out_dir = Path('/kaggle/working/qwen_vl_dpo_v2_tuned_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'Winning adapter directory {out_dir} contents: {files}')
assert (out_dir / 'adapter_config.json').exists(), 'adapter_config.json missing -- sweep did not save a winning adapter.'

winning_config_path = out_dir / 'winning_config.json'
if winning_config_path.exists():
    print('Winning config:')
    print(json.dumps(json.loads(winning_config_path.read_text(encoding='utf-8')), indent=2))

sweep_results_path = Path('/kaggle/working/dpo_sweep/sweep_results.json')
if sweep_results_path.exists():
    print('\nAll config summaries:')
    for r in json.loads(sweep_results_path.read_text(encoding='utf-8')):
        print(f"  {r['config']}: {r['summary']}")
